# 05 — Predictive Modeling Notebook

**Owner:** M5 — Thư  
**Purpose:** run/review the M5 model pipeline and prepare the PSM propensity-score handoff for M6.

This notebook is organized into **2 layers**:

```text
LAYER 1 — Core M5 Modeling
Run/review churn model, calibration, value model, expected profit, and high-risk candidate outputs.

LAYER 2 — PSM Support for M6
Run/review the Propensity Score model after M4 provides treatment flags.
```

Core reusable logic stays in scripts:

```text
scripts/modeling.py               -> full M5 churn/value/profit pipeline
scripts/psm_propensity_score.py   -> PSM propensity-score support for M6
```

## How to use this notebook

### Recommended order

1. Run **Setup**.
2. Run **Layer 1** if you need to regenerate or review M5 outputs.
3. Run **Layer 2** only after M4 provides the real PSM treatment flag file.

### Important switches

All expensive / final steps are disabled by default:

```python
RUN_FULL_M5_PIPELINE = False
RUN_PSM_PROPENSITY = False
```

Change them to `True` only when you intentionally want to rerun that step.

## 0. Setup paths and imports

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').exists() and (PROJECT_ROOT.parent / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = (PROJECT_ROOT / 'config' / 'paths.yaml').resolve()
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = MODELS_DIR / 'reports'
M6_HANDOFF_DIR = MODELS_DIR / 'm6_handoff'
DIAGNOSTICS_DIR = MODELS_DIR / 'diagnostics'
ARTIFACTS_DIR = MODELS_DIR / 'artifacts'
PSM_INPUTS_DIR = MODELS_DIR / 'psm_inputs'

print('Project root:', PROJECT_ROOT)
print('Config:', CONFIG_PATH)
print('Models:', MODELS_DIR)

Project root: C:\Users\Thuww\DDM-Churn-Project
Config: C:\Users\Thuww\DDM-Churn-Project\config\paths.yaml
Models: C:\Users\Thuww\DDM-Churn-Project\models


# LAYER 1 — Core M5 Modeling Pipeline

This layer is the main M5 model workflow. It answers:

```text
1. Which households are likely to churn?
2. Are churn probabilities calibrated enough for business formulas?
3. If a household is retained/active, how much short-term value can it generate?
4. Under business assumptions, does treatment have positive expected profit?
5. Which customers should be passed to M6 as high-risk candidates?
```

This layer is handled by:

```bash
python scripts/modeling.py --config config/paths.yaml
```

## 1.1 Validate core M5 inputs

M5's primary input is the M4 household-level feature table:

```text
models/final_ML_features.csv
```

This table should contain one row per household, the churn label, and feature columns. M5 does **not** rebuild M4 feature engineering here.

In [2]:
from scripts.data_processing import validate_core_inputs

input_checks = validate_core_inputs(CONFIG_PATH)
input_checks

{
  "feature_table_csv": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv",
    "exists": true,
    "size_bytes": 696521
  },
  "transaction_master_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet",
    "exists": true,
    "size_bytes": 24336697
  },
  "customer_base_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet",
    "exists": true,
    "size_bytes": 122150
  }
}


{'feature_table_csv': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv',
  'exists': True,
  'size_bytes': 696521},
 'transaction_master_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet',
  'exists': True,
  'size_bytes': 24336697},
 'customer_base_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet',
  'exists': True,
  'size_bytes': 122150}}

## 1.2 Run the full core M5 pipeline

This cell explicitly runs:

```bash
python scripts/modeling.py --config config/paths.yaml
```

It regenerates the core M5 outputs:

```text
models/reports/       -> model metrics, calibration, top-K, profit summaries
models/m6_handoff/    -> high-risk / priority customer files for M6
models/diagnostics/   -> calibration, SHAP, seasonality, audit files
models/artifacts/     -> model artifact package
```

Leave `RUN_FULL_M5_PIPELINE = False` if you only want to review existing outputs.

In [ ]:
RUN_FULL_M5_PIPELINE = True
if RUN_FULL_M5_PIPELINE:
    import subprocess
    config_path = (PROJECT_ROOT / 'config' / 'paths.yaml').resolve()
    cmd = [
        sys.executable,
        str((PROJECT_ROOT / 'scripts' / 'modeling.py').resolve()),
        '--config',
        str(config_path),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)

summary_path = PROJECT_ROOT / 'reports' / 'internal_briefs' / 'm5_pipeline_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Loaded pipeline summary from:', summary_path.relative_to(PROJECT_ROOT))
else:
    summary = {}
    print('No pipeline summary found yet. Set RUN_FULL_M5_PIPELINE=True to run M5.')

summary

Running: c:\Users\Thuww\AppData\Local\Programs\Python\Python312\python.exe C:\Users\Thuww\DDM-Churn-Project\scripts\modeling.py --config C:\Users\Thuww\DDM-Churn-Project\config\paths.yaml
Loaded pipeline summary from: reports\internal_briefs\m5_pipeline_summary.json


{'version': 'v3_discounted_two_part_value_shap_seasonality',
 'project_root': 'C:\\Users\\Thuww\\DDM-Churn-Project',
 'feature_rows': 2499,
 'churn_rate': 0.1208483393357343,
 'champion_churn_model': 'Logistic Regression balanced',
 'calibration_method': 'isotonic',
 'champion_threshold': 0.06999999999999999,
 'test_PR_AUC_calibrated': 0.3185082366992907,
 'test_F2_score_calibrated': 0.4721030042918455,
 'test_brier_score_calibrated': 0.09326363012585626,
 'active_model': 'Active Random Forest__isotonic',
 'value_model': 'Random Forest Regressor',
 'annual_discount_rate': 0.08,
 'base_profitable_customers': 0,
 'shap_status_file': 'C:\\Users\\Thuww\\DDM-Churn-Project\\reports\\internal_briefs\\M5_shap_status.json',
 'outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models',
 'reports_outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\reports',
 'm6_handoff_outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\m6_handoff',
 'diagnostics_outputs_dir': 'C:\\Users\\Thuww\

## 1.3 Load core M5 output tables

These helper functions read outputs from the organized M5 folders. They also fall back to `models/` root for older runs.

In [4]:
def read_m5_output(filename: str, folder: Path):
    candidates = [folder / filename, MODELS_DIR / filename]
    for path in candidates:
        if path.exists():
            print(f'Loaded: {path.relative_to(PROJECT_ROOT)}')
            return pd.read_csv(path)
    print(f'Missing: {filename}')
    return pd.DataFrame()

def show_selected(df: pd.DataFrame, columns: list[str], n: int = 10):
    if df.empty:
        return df
    available = [c for c in columns if c in df.columns]
    return df[available].head(n)

model_metrics = read_m5_output('model_metrics.csv', REPORTS_DIR)
calibration_summary = read_m5_output('calibration_summary.csv', REPORTS_DIR)
champion_test_metrics = read_m5_output('champion_test_metrics.csv', REPORTS_DIR)
active_model_metrics = read_m5_output('active_model_metrics.csv', DIAGNOSTICS_DIR)
value_model_metrics = read_m5_output('value_model_metrics.csv', REPORTS_DIR)
top_k_precision = read_m5_output('top_k_precision_summary.csv', REPORTS_DIR)
ranking_deciles = read_m5_output('ranking_decile_performance.csv', REPORTS_DIR)
scenario_profit = read_m5_output('scenario_profit_summary.csv', REPORTS_DIR)
profit_threshold = read_m5_output('profit_threshold_analysis.csv', REPORTS_DIR)


Loaded: models\reports\model_metrics.csv
Loaded: models\reports\calibration_summary.csv
Loaded: models\reports\champion_test_metrics.csv
Loaded: models\diagnostics\active_model_metrics.csv
Loaded: models\reports\value_model_metrics.csv
Loaded: models\reports\top_k_precision_summary.csv
Loaded: models\reports\ranking_decile_performance.csv
Loaded: models\reports\scenario_profit_summary.csv
Loaded: models\reports\profit_threshold_analysis.csv


## 1.4 Churn model benchmark

File:

```text
models/reports/model_metrics.csv
```

Purpose:

> Compare candidate churn classifiers before final business interpretation.

Key interpretation:

- `PR-AUC`: main ranking metric for imbalanced churn.
- `F2_score`: favors recall over precision.
- `precision`: among predicted churn-risk households, how many churned.
- `recall`: among actual churners, how many were caught.
- `Brier score`: probability quality.

Because churn rate is low, global precision can look low. For M6, **top-K precision and risk ranking** matter more than global precision.

In [5]:
show_selected(
    model_metrics,
    [
        'model', 'tuned', 'features_used', 'best_cv_PR_AUC', 'best_cv_PR_AUC_std',
        'val_PR_AUC', 'val_ROC_AUC', 'val_F2_score', 'val_precision', 'val_recall',
        'test_PR_AUC', 'test_F2_score', 'test_precision', 'test_recall', 'test_brier_score'
    ],
    n=10,
)

,model,tuned,features_used,best_cv_PR_AUC,best_cv_PR_AUC_std,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,test_PR_AUC,test_F2_score,test_precision,test_recall,test_brier_score
0,Logistic Regression balanced,True,numeric+categorical,0.315951,0.045084,0.370078,0.746443,0.483559,0.183150,0.819672,0.342068,0.457380,0.182573,0.733333,0.205234
1,XGBoost weighted,True,numeric+categorical,0.371115,0.053997,0.363531,0.743773,0.506024,0.245614,0.688525,0.364331,0.460199,0.228395,0.616667,0.169667
2,Extra Trees balanced,True,numeric+categorical,0.361423,0.062012,0.351211,0.743792,0.496689,0.215311,0.737705,0.368152,0.433790,0.191919,0.633333,0.110490
3,Random Forest balanced,True,numeric+categorical,0.362261,0.067781,0.301253,0.728929,0.498221,0.176101,0.918033,0.365378,0.468468,0.165079,0.866667,0.099823
4,Dummy prior,False,none,NaN,NaN,0.122000,0.500000,0.409946,0.122000,1.000000,0.120000,0.405405,0.120000,1.000000,0.105601


## 1.5 Calibration summary

File:

```text
models/reports/calibration_summary.csv
```

Purpose:

> Compare raw, sigmoid, and isotonic probabilities for the selected churn model.

Why it matters:

M5 uses churn probability directly in expected-profit formulas. Therefore, raw model probability should not be used if it is miscalibrated.

Interpretation:

- `raw_uncalibrated`: original model probability.
- `sigmoid`: smoother calibrated probability.
- `isotonic`: flexible calibration, but may create flat spots.
- `selected_for_champion = True`: final calibrated probability used as `p_churn_calibrated`.

In [6]:
show_selected(
    calibration_summary,
    [
        'champion_model', 'calibration_method', 'val_PR_AUC', 'val_ROC_AUC',
        'val_F2_score', 'val_precision', 'val_recall', 'val_brier_score',
        'val_threshold', 'test_PR_AUC', 'test_F2_score', 'test_precision', 'test_recall',
        'test_brier_score', 'test_mean_predicted_probability', 'test_actual_positive_rate',
        'test_calibration_gap_mean_minus_actual', 'val_brier_gap_from_best',
        'calibration_selection_eligible', 'selected_for_champion'
    ],
    n=10,
)

,champion_model,calibration_method,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,val_brier_score,val_threshold,test_PR_AUC,test_F2_score,test_precision,test_recall,test_brier_score,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,val_brier_gap_from_best,calibration_selection_eligible,selected_for_champion
0,Logistic Regression balanced,isotonic,0.372438,0.771761,0.501002,0.196078,0.819672,0.087827,0.07,0.318508,0.472103,0.194690,0.733333,0.093264,0.121285,0.12,0.001285,0.000000,True,True
1,Logistic Regression balanced,sigmoid,0.370078,0.746443,0.477528,0.175862,0.836066,0.096087,0.10,0.342068,0.452756,0.171642,0.766667,0.094887,0.121580,0.12,0.001580,0.008260,True,False
2,Logistic Regression balanced,raw_uncalibrated,0.370078,0.746443,0.483559,0.183150,0.819672,0.206087,0.43,0.342068,0.457380,0.182573,0.733333,0.205234,0.454084,0.12,0.334084,0.118259,False,False


## 1.6 Active model and value model

Files:

```text
models/diagnostics/active_model_metrics.csv
models/reports/value_model_metrics.csv
```

These are part of the **value layer**, not the main churn model.

### Active model

Estimates:

```text
p_future_active = P(household generates revenue in the 60-day prediction window)
```

This is only a supporting component, not an independent targeting signal.

### Value model

Estimates:

```text
predicted_discounted_value_60d_if_active
```

This is **discounted 60-day value**, not full lifetime CLV.

In [7]:
print('Active model metrics')
display(show_selected(
    active_model_metrics,
    [
        'active_model', 'calibration_method', 'val_PR_AUC', 'val_brier_score',
        'val_threshold', 'test_brier_score', 'test_mean_predicted_probability',
        'test_actual_positive_rate', 'test_calibration_gap_mean_minus_actual',
        'test_TN', 'test_FP', 'test_FN', 'test_TP'
    ],
    n=10,
))

print('Value model metrics')
display(show_selected(
    value_model_metrics,
    [
        'value_model', 'target', 'val_RMSE_log', 'val_MAE_log', 'val_R2_log',
        'val_MAE_revenue', 'test_RMSE_log', 'test_MAE_log', 'test_R2_log',
        'test_MAE_revenue'
    ],
    n=10,
))

Active model metrics


,active_model,calibration_method,val_PR_AUC,val_brier_score,val_threshold,test_brier_score,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP
0,Active Random Forest,isotonic,0.985166,0.060430,0.01,0.071452,0.901050,0.892,0.009050,0,54,0,446
1,Active Logistic Regression,isotonic,0.983192,0.061666,0.01,0.077036,0.902308,0.892,0.010308,0,54,0,446
2,Active Random Forest,raw_uncalibrated,0.985167,0.064531,0.47,0.070937,0.888970,0.892,-0.003030,0,54,1,445
3,Active Logistic Regression,raw_uncalibrated,0.983710,0.068930,0.12,0.072657,0.890872,0.892,-0.001128,0,54,0,446
4,Active Random Forest,sigmoid,0.985167,0.069569,0.61,0.078481,0.904744,0.892,0.012744,0,54,1,445
5,Active Logistic Regression,sigmoid,0.983710,0.071183,0.37,0.081314,0.907177,0.892,0.015177,0,54,0,446


Value model metrics


,value_model,target,val_RMSE_log,val_MAE_log,val_R2_log,val_MAE_revenue,test_RMSE_log,test_MAE_log,test_R2_log,test_MAE_revenue
0,Random Forest Regressor,log1p(discounted_future_revenue_60d | active),0.881957,0.655148,0.515687,138.864944,0.885131,0.641719,0.497328,143.768862
1,XGBoost Regressor,log1p(discounted_future_revenue_60d | active),0.917339,0.671099,0.476049,143.241625,0.912310,0.660565,0.465984,149.548866
2,Ridge Regression,log1p(discounted_future_revenue_60d | active),0.918751,0.686814,0.474434,166.306744,0.957231,0.713288,0.412100,191.176740


## 1.7 Top-K ranking and profit scenarios

Files:

```text
models/reports/top_k_precision_summary.csv
models/reports/scenario_profit_summary.csv
models/reports/profit_threshold_analysis.csv
```

Purpose:

> Show whether M5 is useful for risk ranking and whether treatment looks profitable under business assumptions.

Expected-profit formula:

```text
expected_profit = p_churn_calibrated × save_rate × value_if_active × gross_margin - treatment_cost
```

If base scenario has zero profitable customers, the correct conclusion is:

> Do not auto-rollout paid vouchers. Use M5 output as candidate ranking for M6 / PSM / future campaign validation.

In [8]:
print('Top-K precision')
display(top_k_precision.head(20))

print('Scenario profit summary')
display(scenario_profit.head(20))

print('Profit threshold analysis')
display(profit_threshold.head(20))

Top-K precision


,score_name,top_k_share,top_k_customer_count,baseline_churn_rate,precision_at_k,lift_vs_baseline,churn_count_at_k,recall_at_k,min_score_in_top_k,mean_score_in_top_k,max_score_in_top_k
0,p_churn_calibrated,0.05,125,0.120848,0.512,4.236715,64,0.211921,0.512382,0.613755,1.000000
1,p_churn_calibrated,0.10,250,0.120848,0.420,3.475430,105,0.347682,0.249415,0.531618,1.000000
2,p_churn_calibrated,0.20,500,0.120848,0.328,2.714146,164,0.543046,0.133333,0.353629,1.000000
3,risk_value_score,0.05,125,0.120848,0.096,0.794384,12,0.039735,58.054140,81.412370,352.213642
4,risk_value_score,0.10,250,0.120848,0.112,0.926781,28,0.092715,47.576101,67.114346,352.213642
5,risk_value_score,0.20,500,0.120848,0.140,1.158477,70,0.231788,32.273943,52.985907,352.213642
6,expected_incremental_profit_base,0.05,125,0.120848,0.096,0.794384,12,0.039735,-4.274323,-3.982345,-0.597329
7,expected_incremental_profit_base,0.10,250,0.120848,0.112,0.926781,28,0.092715,-4.405299,-4.161071,-0.597329
8,expected_incremental_profit_base,0.20,500,0.120848,0.140,1.158477,70,0.231788,-4.596576,-4.337676,-0.597329


Scenario profit summary


,scenario,scenario_type,gross_margin,save_rate_given_treatment,treatment_cost,profitable_customer_count,profitable_customer_share,total_expected_incremental_profit_if_target_positive_only,top_30pct_risk_customer_count,total_expected_incremental_profit_if_target_top_30pct_risk
0,conservative,named,0.20,0.03,5.0,0,0.000000,0.000000,750,-3628.225549
1,base,named,0.25,0.05,5.0,0,0.000000,0.000000,750,-3496.303227
2,optimistic,named,0.30,0.08,3.0,6,0.002401,11.009974,750,-1762.902195


Profit threshold analysis


,scenario,expected_profit_column,selection_rule,selected_customer_count,selected_customer_share,baseline_churn_rate,selected_churn_rate,lift_vs_baseline,total_expected_incremental_profit,mean_expected_incremental_profit,min_expected_incremental_profit,max_expected_incremental_profit,min_p_churn_calibrated,mean_p_churn_calibrated,max_p_churn_calibrated
0,conservative,expected_incremental_profit_conservative,profit_positive,0,0.000000,0.120848,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,conservative,expected_incremental_profit_conservative,top_5pct_by_expected_profit,125,0.050020,0.120848,0.096,0.794384,-563.940722,-4.511526,-4.651675,-2.886718,0.050000,0.157255,0.666667
2,conservative,expected_incremental_profit_conservative,top_10pct_by_expected_profit,250,0.100040,0.120848,0.112,0.926781,-1149.328482,-4.597314,-4.714543,-2.886718,0.050000,0.158616,0.666667
3,conservative,expected_incremental_profit_conservative,top_20pct_by_expected_profit,500,0.200080,0.120848,0.140,1.158477,-2341.042280,-4.682085,-4.806356,-2.886718,0.050000,0.172644,1.000000
4,base,expected_incremental_profit_base,profit_positive,0,0.000000,0.120848,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
5,base,expected_incremental_profit_base,top_5pct_by_expected_profit,125,0.050020,0.120848,0.096,0.794384,-497.793171,-3.982345,-4.274323,-0.597329,0.050000,0.157255,0.666667
6,base,expected_incremental_profit_base,top_10pct_by_expected_profit,250,0.100040,0.120848,0.112,0.926781,-1040.267670,-4.161071,-4.405299,-0.597329,0.050000,0.158616,0.666667
7,base,expected_incremental_profit_base,top_20pct_by_expected_profit,500,0.200080,0.120848,0.140,1.158477,-2168.838084,-4.337676,-4.596576,-0.597329,0.050000,0.172644,1.000000
8,optimistic,expected_incremental_profit_optimistic,profit_positive,6,0.002401,0.120848,0.000,0.000000,11.009974,1.834996,0.084137,5.453127,0.111801,0.350425,0.545455
9,optimistic,expected_incremental_profit_optimistic,top_5pct_by_expected_profit,125,0.050020,0.120848,0.096,0.794384,-130.762889,-1.046103,-1.606701,5.453127,0.050000,0.157255,0.666667


## 1.8 Core handoff files for M6

Folder:

```text
models/m6_handoff/
```

Important files:

| File | Purpose |
|---|---|
| `high_risk_customers_for_ab_test.csv` | Top-risk candidate pool. Can also support PSM subgroup analysis. |
| `priority_customers_all.csv` | Full ranking with risk/value/profit fields. |
| `churn_predictions.csv` | Full customer-level scoring output. |

These files identify high-risk households. They do **not** prove coupon treatment works.

In [9]:
high_risk = read_m5_output('high_risk_customers_for_ab_test.csv', M6_HANDOFF_DIR)
priority_all = read_m5_output('priority_customers_all.csv', M6_HANDOFF_DIR)

print('High-risk customer file shape:', high_risk.shape)
display(show_selected(
    high_risk,
    [
        'household_key', 'p_churn_calibrated', 'risk_rank', 'risk_decile',
        'risk_value_score', 'risk_value_rank', 'priority_segment',
        'predicted_discounted_value_60d_if_active',
        'expected_incremental_profit_base', 'recommended_treatment_action_base'
    ],
    n=10,
))

Loaded: models\m6_handoff\high_risk_customers_for_ab_test.csv
Loaded: models\m6_handoff\priority_customers_all.csv
High-risk customer file shape: (750, 65)


,household_key,p_churn_calibrated,risk_rank,risk_decile,risk_value_score,risk_value_rank,priority_segment,predicted_discounted_value_60d_if_active,expected_incremental_profit_base,recommended_treatment_action_base
0,2072,1.000000,1,1,35.264844,430,High Risk - Low Value,35.264844,-4.559189,A/B test candidate only; not profitable under ...
1,313,1.000000,2,1,23.356959,806,High Risk - Low Value,23.356959,-4.708038,A/B test candidate only; not profitable under ...
2,1504,0.980231,3,1,41.096982,334,High Risk - Low Value,41.925799,-4.486288,A/B test candidate only; not profitable under ...
3,1183,0.670922,4,1,28.684267,612,High Risk - Low Value,42.753509,-4.641447,A/B test candidate only; not profitable under ...
4,102,0.666667,5,1,46.541172,261,High Risk - Low Value,69.811758,-4.418235,A/B test candidate only; not profitable under ...
5,1038,0.666667,6,1,28.709841,610,High Risk - Low Value,43.064762,-4.641127,A/B test candidate only; not profitable under ...
6,1052,0.666667,7,1,36.071463,418,High Risk - Low Value,54.107194,-4.549107,A/B test candidate only; not profitable under ...
7,11,0.666667,8,1,25.817101,715,High Risk - Low Value,38.725652,-4.677286,A/B test candidate only; not profitable under ...
8,1122,0.666667,9,1,34.565237,444,High Risk - Low Value,51.847856,-4.567935,A/B test candidate only; not profitable under ...
9,1162,0.666667,10,1,24.821281,745,High Risk - Low Value,37.231922,-4.689734,A/B test candidate only; not profitable under ...


# LAYER 2 — PSM Support for M6

This layer is separate from the core M5 churn model.

M6 has chosen **Propensity Score Matching (PSM)**. M5's support is only to generate:

```text
propensity_score = P(is_treated = 1 | observed covariates)
```

M6 will use this score for:

```text
matching treated/control households
balance plots
effect estimation
ROI / ROMI calculation
```

M5 does **not** perform final matching or ROI calculation in this notebook.

## 2.1 PSM input contract with M4


In [10]:
psm_flag_path = PSM_INPUTS_DIR / 'psm_treatment_flags.csv'
psm_template_path = PSM_INPUTS_DIR / 'psm_treatment_flags_TEMPLATE.csv'
psm_output_path = M6_HANDOFF_DIR / 'propensity_scores_for_psm.csv'

print('Expected PSM input from M4:', psm_flag_path.relative_to(PROJECT_ROOT))
print('PSM output for M6:', psm_output_path.relative_to(PROJECT_ROOT))

if psm_flag_path.exists():
    flags_preview = pd.read_csv(psm_flag_path)
    print('Treatment flag file found:', flags_preview.shape)
    display(flags_preview.head())
else:
    print('Treatment flag file not found yet.')
    print('Ask M4 to create:', psm_flag_path.relative_to(PROJECT_ROOT))
    if psm_template_path.exists():
        print('Template exists:', psm_template_path.relative_to(PROJECT_ROOT))

Expected PSM input from M4: models\psm_inputs\psm_treatment_flags.csv
PSM output for M6: models\m6_handoff\propensity_scores_for_psm.csv
Treatment flag file found: (2499, 4)


,household_key,is_treated,treatment_source,treatment_cutoff_day
0,1,1,Both,651
1,10,1,Mailer_Only,651
2,100,1,Both,651
3,1000,1,Both,651
4,1001,1,Both,651


## 2.2 Run the PSM propensity-score model

This step runs:

```bash
python scripts/psm_propensity_score.py --config config/paths.yaml
```

It fits a simple Logistic Regression model:

```text
propensity_score = P(is_treated = 1 | covariates)
```

The script excludes unsafe columns such as outcome/future/profit/coupon-like variables to reduce leakage/collider-bias risk.

Set `RUN_PSM_PROPENSITY = True` 

In [11]:
RUN_PSM_PROPENSITY = True
ALLOW_DUMMY_PSM = False

if RUN_PSM_PROPENSITY:
    import subprocess
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'psm_propensity_score.py'),
        '--config', str(CONFIG_PATH),
    ]
    if ALLOW_DUMMY_PSM:
        cmd.append('--allow-dummy')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)
else:
    print('Skipped PSM propensity score generation. Set RUN_PSM_PROPENSITY=True after M4 provides real treatment flags.')

Running: c:\Users\Thuww\AppData\Local\Programs\Python\Python312\python.exe C:\Users\Thuww\DDM-Churn-Project\scripts\psm_propensity_score.py --config C:\Users\Thuww\DDM-Churn-Project\config\paths.yaml


## 2.4 Review PSM output for M6

Final handoff file:

```text
models/m6_handoff/propensity_scores_for_psm.csv
```

Minimum columns:

| Column | Meaning |
|---|---|
| `household_key` | Household identifier |
| `is_treated` | Treatment flag from M4 |
| `propensity_score` | Estimated probability of being treated |
| `propensity_logit` | Logit transform, useful for caliper matching |
| `common_support_flag` | Whether household falls inside treated/control common support |

M6 uses this file for matching and balance plots.

In [12]:
psm_scores = read_m5_output('propensity_scores_for_psm.csv', M6_HANDOFF_DIR)
if not psm_scores.empty:
    display(show_selected(
        psm_scores,
        [
            'household_key', 'is_treated', 'propensity_score', 'propensity_logit',
            'common_support_flag', 'p_churn_calibrated', 'risk_rank', 'risk_decile',
            'priority_segment', 'predicted_discounted_value_60d_if_active'
        ],
        n=10,
    ))
else:
    print('No PSM score output yet. Run the PSM script after M4 provides real treatment flags.')

Loaded: models\m6_handoff\propensity_scores_for_psm.csv


,household_key,is_treated,propensity_score,propensity_logit,common_support_flag,p_churn_calibrated,risk_rank,risk_decile,priority_segment,predicted_discounted_value_60d_if_active
0,1,1,0.996643,5.693387,False,0.000000,2236,9,Low Risk - High Value,358.332546
1,10,1,0.950797,2.961356,True,0.250000,234,1,High Risk - Low Value,68.907404
2,100,1,0.997085,5.834829,False,0.111801,520,3,High Risk - Low Value,208.441796
3,1000,1,0.999415,7.443768,False,0.062500,1232,5,Low Risk - High Value,345.026452
4,1001,1,0.996106,5.544353,False,0.062500,1233,5,Low Risk - High Value,377.227402
5,1002,1,0.994107,5.128019,False,0.000000,2237,9,Low Risk - Low Value,130.877428
6,1003,1,0.998888,6.800543,False,0.235294,260,2,High Risk - Low Value,80.796737
7,1004,1,0.998093,6.260109,False,0.111801,521,3,High Risk - Low Value,183.754465
8,1005,1,0.999573,7.758625,False,0.133333,354,2,High Risk - Low Value,146.783063
9,1006,1,0.976670,3.734415,True,0.062500,1234,5,Low Risk - Low Value,32.436152
